# Content-Based Recommender Using Course Similarity

This notebook implements the course-similarity recommender described in the presentation. It uses the precomputed course-course similarity matrix and recommends unseen courses that are very similar to a course already associated with the user.

In [1]:
from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np

DATA_DIR = Path("datasets")
DATA_URLS = {
    "ratings.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/ratings.csv",
    "course_genre.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_genre.csv",
    "rs_content_test.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/rs_content_test.csv",
    "user_profile.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/user_profile.csv",
    "course_processed.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_processed.csv",
    "courses_bows.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/courses_bows.csv",
    "sim.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/sim.csv",
}

def ensure_dataset(filename):
    DATA_DIR.mkdir(exist_ok=True)
    path = DATA_DIR / filename
    if not path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(DATA_URLS[filename], path)
    return path

def load_csv(filename, **kwargs):
    return pd.read_csv(ensure_dataset(filename), **kwargs)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

In [2]:
course_df = load_csv("course_processed.csv")
sim_matrix = load_csv("sim.csv").to_numpy()
test_users_df = load_csv("rs_content_test.csv")

course_ids = course_df["COURSE_ID"].tolist()
id_to_idx = {course_id: idx for idx, course_id in enumerate(course_ids)}
all_courses = set(course_ids)

print("Courses:", course_df.shape)
print("Similarity matrix:", sim_matrix.shape)
print("Test interactions:", test_users_df.shape)

Courses: (307, 3)
Similarity matrix: (307, 307)
Test interactions: (9402, 3)


## Recommendation Logic

The presentation uses a similarity threshold of `0.6`. To reproduce its evaluation style, each test user is represented by one held-out/reference course from `rs_content_test.csv`, then unseen courses above the similarity threshold are recommended.

In [4]:
SIMILARITY_THRESHOLD = 0.6

def recommend_similar_courses(reference_course_ids, already_seen_course_ids, top_n=20):
    scores = {}
    candidate_courses = all_courses.difference(already_seen_course_ids)

    for reference_course_id in reference_course_ids:
        if reference_course_id not in id_to_idx:
            continue
        reference_idx = id_to_idx[reference_course_id]

        for candidate_course_id in candidate_courses:
            candidate_idx = id_to_idx[candidate_course_id]
            score = float(sim_matrix[reference_idx, candidate_idx])
            if score >= SIMILARITY_THRESHOLD:
                scores[candidate_course_id] = max(score, scores.get(candidate_course_id, 0.0))

    ranked = sorted(scores.items(), key=lambda pair: pair[1], reverse=True)[:top_n]
    return ranked

sample_user = int(test_users_df["user"].iloc[0])
sample_reference_courses = test_users_df.loc[test_users_df["user"] == sample_user, "item"].head(1).tolist()
sample_seen = set(test_users_df.loc[test_users_df["user"] == sample_user, "item"])
recommend_similar_courses(sample_reference_courses, sample_seen)

[]

In [5]:
user_reference_df = test_users_df.groupby("user").max().reset_index()

rows = []
for _, row in user_reference_df.iterrows():
    user_id = row["user"]
    reference_course = row["item"]
    seen_courses = set(test_users_df.loc[test_users_df["user"] == user_id, "item"])
    for course_id, score in recommend_similar_courses([reference_course], seen_courses):
        rows.append({"USER": user_id, "COURSE_ID": course_id, "SCORE": score})

similarity_recommendations_df = pd.DataFrame(rows)
average_per_user = len(similarity_recommendations_df) / user_reference_df["user"].nunique()

print("Total generated recommendations:", len(similarity_recommendations_df))
print(f"Average recommendations per test user: {average_per_user:.3f}")
display(similarity_recommendations_df.head(10))

Total generated recommendations: 966
Average recommendations per test user: 0.966


,USER,COURSE_ID,SCORE
0,52091,ML0120ENv3,0.982873
1,85625,TMP0101EN,0.889499
2,85625,TA0105EN,0.659829
3,85625,BD0151EN,0.630470
4,108541,excourse22,0.647502
5,108541,excourse62,0.647502
6,109915,WA0103EN,0.631153
7,149690,TMP0101EN,0.889499
8,149690,TA0105EN,0.659829
9,149690,BD0151EN,0.630470


In [6]:
top_similarity_recommendations = (
    similarity_recommendations_df["COURSE_ID"]
    .value_counts()
    .head(10)
    .rename_axis("COURSE_ID")
    .reset_index(name="recommendation_count")
    .merge(course_df[["COURSE_ID", "TITLE"]], on="COURSE_ID", how="left")
)

display(top_similarity_recommendations)

,COURSE_ID,recommendation_count,TITLE
0,excourse22,257,introduction to data science in python
1,excourse62,257,introduction to data science in python
2,WA0103EN,101,watson analytics for social media
3,TA0105,41,text analytics 101
4,DS0110EN,38,data science with open data
5,excourse46,24,machine learning
6,excourse47,24,machine learning for all
7,excourse63,23,a crash course in data science
8,excourse65,23,data science fundamentals for data analysts
9,TMP0101EN,17,text analysis


## Interpretation

This method is conservative. It usually recommends fewer courses because it requires strong course-to-course similarity. It is best for "more like this" use cases where precision matters more than coverage.